# Unit 6, Lecture 3: Authentication, Authorization, Auditing

Lecture 2 ended on least privilege: an agent should only do what its job needs.
That raises the question it could not answer: **who is asking, and how does the
system know?** This lecture supplies the machinery.

Three questions every secure system must answer, as a building with a guard and
keycards:

- **Authentication**, who are you? (the guard checks your ID)
- **Authorization**, what may you do? (your keycard opens only certain rooms)
- **Auditing**, what happened? (every swipe is logged)

**McDonald's McHire (2025)** failed the first two at once, exposing 64 million
applicants: the login accepted `123456` (authentication), and changing an id in
the request returned any record (authorization, an IDOR bug). Each pillar below
closes one of these. All offline.

## 1. Authentication: store a salted hash, never the password

Because databases leak. A stolen hash is useless; a stolen password is every
account.

In [ ]:
from cse476.auth import hash_password, verify_password

salt, stored = hash_password("correct-horse-battery")
print("stored is a hash, not the password:", stored[:24], "...")
print("right password:", verify_password("correct-horse-battery", salt, stored))
print("wrong password:", verify_password("123456", salt, stored))

salt2, stored2 = hash_password("correct-horse-battery")
print("same password hashes differently:", stored != stored2)

### Reject the password that sank McHire

In [ ]:
from cse476.auth import is_weak_password

for pw in ["123456", "password", "abc12", "correct-horse-battery"]:
    print(f"{pw:24} weak? {is_weak_password(pw)}")

## 2. Authorization: check ownership on every request

Being logged in is not permission. The McHire IDOR bug: any logged-in user could
read any record by changing an id.

In [ ]:
from cse476.auth import can_access

print("alice reads her own record :", can_access("alice", "alice"))
print("alice reads bob's record   :", can_access("alice", "bob"))       # the IDOR
print("admin reads any record     :", can_access("alice", "bob", role="admin"))

### Roles: least privilege, made enforceable

In [ ]:
from cse476.auth import Role, require_role

viewer    = Role("viewer",    frozenset({"read"}))
recruiter = Role("recruiter", frozenset({"read", "contact"}))

print("viewer may read      :", require_role(viewer, "read"))
print("viewer may contact   :", require_role(viewer, "contact"))     # not granted
print("recruiter may contact:", require_role(recruiter, "contact"))

## 3. Auditing: an append-only trail of who did what

Log allowed and denied actions. The denied ones matter most: a burst of denied
reads of other people's records is an IDOR attack in progress.

In [ ]:
from cse476.auth import AuditLog

log = AuditLog()
log.record("alice", "read", "record:alice", allowed=True)
log.record("alice", "read", "record:bob",   allowed=False)   # IDOR attempt
log.record("alice", "read", "record:carol", allowed=False)

print("total entries :", len(log.entries))
print("denied        :", [(e.who, e.target) for e in log.denied()])
print("alice did     :", len(log.by_user("alice")), "things")

## Why you need all three

In [ ]:
from cse476.auth import AUTH_MAP, why_all_three

for k, v in AUTH_MAP.items():
    print(f"{k:32} ->  {v}")
print()
for k, v in why_all_three().items():
    print(f"{k:22}: {v}")

The three are a chain: a break in any link fails the system. McHire had a broken
first link (weak password) and a missing second (no ownership check), so the third
could only have recorded the disaster, not prevented it.

In production you use **Microsoft Entra ID** (authentication), **role-based access
control** (authorization), and **Azure Monitor** (auditing) rather than
hand-rolling this. Building it by hand today is how you learn to use the platform
correctly.

## Your turn

**1. Hash a password properly.** Make sure any login code stores a salted hash,
never plaintext, and rejects the common-password list.

**2. Add the ownership check.** Find one place your app fetches a record by id, and
check the record belongs to the caller. That one line is the IDOR fix.

**3. What would your audit log show?** If someone attacked today, could you prove
afterwards who accessed what?

In [ ]:
# your work here
